In [1]:
import os
import sys
import numpy as np
import random
import time
from pathlib import Path
import psutil
from pynvml import nvmlInit, nvmlDeviceGetHandleByIndex, nvmlDeviceGetMemoryInfo, nvmlDeviceGetUtilizationRates

# Importación de librerías para manejo de imágenes y transformaciones
from torchvision import transforms
from PIL import Image

# Importación de librerías de deep learning
import torch
from src.models.convolutional_autoencoder_model.model import ConvolutionalAutoencoder

In [2]:
# Inicializar NVML para la GPU
nvmlInit()
device_handle = None
if torch.cuda.is_available():
    device_handle = nvmlDeviceGetHandleByIndex(0)

In [3]:
# Si tenemos disponible GPU, lo usamos
if torch.cuda.is_available():
    device = "cuda"
    torch.cuda.synchronize()  # Asegurar que las operaciones previas en CUDA se hayan completado
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

In [4]:
# Me aseguro de que el directorio raíz del proyecto esté en el sys.path
project_root = Path(os.path.abspath(""))


In [5]:
# Añado el directorio raíz al sys.path si no está ya presente
if project_root not in sys.path:
    sys.path.append(str(project_root))


In [6]:
# Importo las funciones de configuración
from src.config import raw_data_dir, interim_data_dir, models_dir, load_config
from src.utils.datasets import CustomDataset

Current working directory: /home/jorge/development/ImageReconstructionDL/notebooks
Loading configuration from /home/jorge/development/ImageReconstructionDL/src/config.yaml


INFO:albumentations.check_version:A new version of Albumentations is available: 1.4.18 (you have 1.4.8). Upgrade using: pip install --upgrade albumentations


In [7]:
# Cargo la configuración 
config = load_config()

Loading configuration from /home/jorge/development/ImageReconstructionDL/src/config.yaml


In [8]:
# Verifico que el dispositivo que se está utilizando
print(f"Usando dispositivo: {device}")


Usando dispositivo: cuda


In [9]:
# Cargar el modelo de autoencoder
encoder_filters = [64, 128, 256, 512, 1024, 2048]
decoder_filters = list(reversed(encoder_filters))
nombre_modelo = "convolutional_autoencoder_model_final_best_hyperparameters_2_2.pth"
models_path = models_dir() / "trained"
model_path = models_path / nombre_modelo
# convierto a string
model_path = str(model_path)

In [10]:
# Inicializar el modelo y cargar los pesos entrenados
modelo = ConvolutionalAutoencoder(encoder_filters, decoder_filters).to(device)
modelo.load_state_dict(torch.load(model_path, map_location=device))
modelo.eval()

ConvolutionalAutoencoder(
  (encoder): Sequential(
    (0): Conv2d(3, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(64, 128, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (3): ReLU()
    (4): Conv2d(128, 256, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (5): ReLU()
    (6): Conv2d(256, 512, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (7): ReLU()
    (8): Conv2d(512, 1024, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (9): ReLU()
    (10): Conv2d(1024, 2048, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (11): ReLU()
  )
  (decoder): Sequential(
    (0): ConvTranspose2d(2048, 1024, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (1): ReLU()
    (2): ConvTranspose2d(1024, 512, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (3): ReLU()
    (4): ConvTranspose2d(512, 256, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (5): ReLU()
    (6): ConvTranspose2d(256, 128, kernel_size=(4, 4

In [11]:
# Ruta de los datos
original_dir = raw_data_dir() / 'reduced_dataset' / 'data'

In [12]:
# Listar todas las imágenes en el directorio (sin usar glob)
image_files = []

In [13]:
# Recorre cada subdirectorio en original_dir
for sub_dir in os.listdir(original_dir):
    sub_dir_path = os.path.join(original_dir, sub_dir)
    if os.path.isdir(sub_dir_path):
        # Añade la ruta completa de cada imagen en el subdirectorio si es .png o .jpg
        image_files.extend(
            [os.path.join(sub_dir_path, f) for f in os.listdir(sub_dir_path) if f.endswith(('.png', '.jpg'))]
        )


In [14]:
# Seleccionar imágenes al azar, verificando que haya suficientes
num_images = min(160, len(image_files))
selected_images = random.sample(image_files, num_images)

In [16]:
# Transformación para las imágenes
transform = transforms.Compose([
    transforms.Resize(256),  # Mantener relación de aspecto
    #transforms.CenterCrop(256),
    transforms.ToTensor(),
])

In [17]:
# Compresión y descompresión de imágenes
compression_times = []
decompression_times = []
compression_cpu_usage = []
decompression_cpu_usage = []
compression_ram_usage = []
decompression_ram_usage = []
compression_gpu_usage = []
decompression_gpu_usage = []
compression_gpu_memory_usage = []
decompression_gpu_memory_usage = []

In [18]:
def get_gpu_metrics():
    if device == "cuda":
        torch.cuda.synchronize()  # Asegurar que todas las operaciones CUDA se hayan completado antes de medir
        gpu_memory = nvmlDeviceGetMemoryInfo(device_handle).used / 1e6  # VRAM en MB
        gpu_utilization = nvmlDeviceGetUtilizationRates(device_handle).gpu  # Utilización de GPU en %
    else:
        gpu_memory = 0
        gpu_utilization = 0
    return gpu_memory, gpu_utilization

In [19]:
for image_name in selected_images:
    # Cargar la imagen original
    image_path = image_name
    image = Image.open(image_path).convert("RGB")

    # Transformar la imagen a tensor
    input_tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        # Medir tiempo de compresión y uso de recursos
        start_time = time.time()
        process = psutil.Process(os.getpid())
        cpu_before = process.cpu_percent(interval=0.1)  # Uso de CPU del proceso
        ram_before = process.memory_info().rss / 1e6  # RAM en MB
        gpu_memory_before, gpu_utilization_before = get_gpu_metrics()

        compressed_data = modelo.encoder(input_tensor)
        compression_time = time.time() - start_time
        compression_times.append(compression_time)

        cpu_after = process.cpu_percent(interval=0.1)
        ram_after = process.memory_info().rss / 1e6  # RAM en MB
        gpu_memory_after, gpu_utilization_after = get_gpu_metrics()

        compression_cpu_usage.append(cpu_after)
        compression_ram_usage.append(ram_after)
        compression_gpu_usage.append(gpu_utilization_after)
        compression_gpu_memory_usage.append(gpu_memory_after)

        # Medir tiempo de descompresión y uso de recursos
        start_time = time.time()
        cpu_before = process.cpu_percent(interval=0.1)
        ram_before = process.memory_info().rss / 1e6  # RAM en MB
        gpu_memory_before, gpu_utilization_before = get_gpu_metrics()

        reconstructed_data = modelo.decoder(compressed_data)
        decompression_time = time.time() - start_time
        decompression_times.append(decompression_time)

        cpu_after = process.cpu_percent(interval=0.1)
        ram_after = process.memory_info().rss / 1e6  # RAM en MB
        gpu_memory_after, gpu_utilization_after = get_gpu_metrics()

        decompression_cpu_usage.append(cpu_after)
        decompression_ram_usage.append(ram_after)
        decompression_gpu_usage.append(gpu_utilization_after)
        decompression_gpu_memory_usage.append(gpu_memory_after)


In [20]:
# Calcular y mostrar los tiempos promedio de compresión y descompresión
average_compression_time = sum(compression_times) / len(compression_times)
average_decompression_time = sum(decompression_times) / len(decompression_times)

In [21]:
# Calcular los consumos promedio de CPU, RAM, GPU y VRAM para compresión y descompresión
average_compression_cpu_usage = sum(compression_cpu_usage) / len(compression_cpu_usage)
average_decompression_cpu_usage = sum(decompression_cpu_usage) / len(decompression_cpu_usage)

average_compression_ram_usage = sum(compression_ram_usage) / len(compression_ram_usage)
average_decompression_ram_usage = sum(decompression_ram_usage) / len(decompression_ram_usage)

average_compression_gpu_usage = sum(compression_gpu_usage) / len(compression_gpu_usage)
average_decompression_gpu_usage = sum(decompression_gpu_usage) / len(decompression_gpu_usage)

average_compression_gpu_memory_usage = sum(compression_gpu_memory_usage) / len(compression_gpu_memory_usage)
average_decompression_gpu_memory_usage = sum(decompression_gpu_memory_usage) / len(decompression_gpu_memory_usage)

In [22]:
print(f"Tiempo promedio de compresión: {average_compression_time:.4f} segundos")
print(f"Tiempo promedio de descompresión: {average_decompression_time:.4f} segundos")

print(f"Uso promedio de CPU durante compresión: {average_compression_cpu_usage:.2f}%")
print(f"Uso promedio de CPU durante descompresión: {average_decompression_cpu_usage:.2f}%")

print(f"Consumo promedio de RAM durante compresión: {average_compression_ram_usage:.2f} MB")
print(f"Consumo promedio de RAM durante descompresión: {average_decompression_ram_usage:.2f} MB")

print(f"Uso promedio de GPU durante compresión: {average_compression_gpu_usage:.2f}%")
print(f"Uso promedio de GPU durante descompresión: {average_decompression_gpu_usage:.2f}%")

print(f"Consumo promedio de VRAM durante compresión: {average_compression_gpu_memory_usage:.2f} MB")
print(f"Consumo promedio de VRAM durante descompresión: {average_decompression_gpu_memory_usage:.2f} MB")


Tiempo promedio de compresión: 0.1024 segundos
Tiempo promedio de descompresión: 0.1018 segundos
Uso promedio de CPU durante compresión: 0.74%
Uso promedio de CPU durante descompresión: 0.81%
Consumo promedio de RAM durante compresión: 995.23 MB
Consumo promedio de RAM durante descompresión: 995.43 MB
Uso promedio de GPU durante compresión: 5.39%
Uso promedio de GPU durante descompresión: 5.42%
Consumo promedio de VRAM durante compresión: 3186.40 MB
Consumo promedio de VRAM durante descompresión: 3186.79 MB
